# Solutions · 08 · Oscillatory rheology

Worked solutions to the exercises of [notebook 08](../notebooks/08_oscillatory_rheology.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engrheo is missing): install it from GitHub
    import engrheo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engrheo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engrheo import datasets, oscillatory
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Using only the 220 degC frequency sweep (unshifted), estimate the terminal slopes of G' and G''. Why are they closer to the theoretical values than at 150 degC?

In [2]:
melt = pd.read_csv(datasets.path("polystyrene_frequency_sweeps.csv"), comment="#")
for T_C in (220, 150):
    g = melt[melt.temperature_C == T_C]
    low = g.omega_rad_s < g.omega_rad_s.min() * 10
    s1 = np.polyfit(np.log(g.omega_rad_s[low]), np.log(g.G_storage_Pa[low]), 1)[0]
    s2 = np.polyfit(np.log(g.omega_rad_s[low]), np.log(g.G_loss_Pa[low]), 1)[0]
    print(f"{T_C} degC, lowest decade: slope of G' {s1:.2f}, of G'' {s2:.2f}   (terminal theory 2 and 1)")

220 degC, lowest decade: slope of G' 1.99, of G'' 1.00   (terminal theory 2 and 1)
150 degC, lowest decade: slope of G' 0.14, of G'' 0.13   (terminal theory 2 and 1)


At 220 degC the melt relaxes about a thousand times faster than at 170 degC, so even the unshifted sweep reaches
deep into the terminal zone and shows slopes near 2 and 1. At 150 degC the same frequencies correspond to the
plateau region, where the slopes are small. The terminal slopes appear only when $\omega\tau_{max} \ll 1$.

## Exercise 2

Determine the gel time using only two frequencies (1 and 31.6 rad/s) instead of four. How robust is the Winter-Chambon estimate?

In [3]:
gel = pd.read_csv(datasets.path("food_gel_time_sweep.csv"), comment="#")
piv = gel.assign(tan_delta=gel.G_loss_Pa / gel.G_storage_Pa).pivot(index="time_min", columns="omega_rad_s", values="tan_delta")
all4 = oscillatory.gel_point(piv.index.to_numpy(), piv.to_numpy())
two = oscillatory.gel_point(piv.index.to_numpy(), piv[[1.0, 31.62]].to_numpy())
print(f"gel time: four frequencies {all4:.2f} min, two frequencies {two:.2f} min (true 23.4)")

gel time: four frequencies 23.38 min, two frequencies 23.38 min (true 23.4)


Even two frequencies locate the crossing of the $\tan\delta$ curves well, because far-apart frequencies give
curves that cross at a large angle. More frequencies add robustness against noise; in practice three to five
are used.

## Exercise 3

**Implement it yourself:** find the crossover of the melt's G' and G'' by hand: locate the sign change of $\ln G' - \ln G''$ and interpolate linearly in log-log coordinates; compare with `oscillatory.crossover`.

In [4]:
melt = pd.read_csv(datasets.path("polystyrene_frequency_sweeps.csv"), comment="#")
g = melt[melt.temperature_C == 170]
w, Gp, Gpp = g.omega_rad_s.to_numpy(), g.G_storage_Pa.to_numpy(), g.G_loss_Pa.to_numpy()
d = np.log(Gp) - np.log(Gpp)
idx = np.nonzero(np.sign(d[:-1]) != np.sign(d[1:]))[0]
if idx.size:
    i = idx[0]; f = d[i] / (d[i] - d[i + 1])
    print(f"by hand: crossover at {np.exp(np.log(w[i]) + f*np.log(w[i+1]/w[i])):.4g} rad/s; library: {oscillatory.crossover(w, Gp, Gpp)[0]:.4g} rad/s")
else:
    print("no crossover within the 170 degC sweep:", oscillatory.crossover(w, Gp, Gpp))

by hand: crossover at 0.1825 rad/s; library: 0.1825 rad/s


The crossover is found from the sign change of $\ln G' - \ln G''$ and a straight-line interpolation in log-log
coordinates - exactly the library's method. If a single sweep does not contain the crossover, the master
curve (which spans more decades) is needed.